# 02 · Data Preprocessing
**CIS 517 – Hotel Booking Cancellation Prediction Using Data Mining**

Goal: clean the raw dataset step by step based on the findings of `01_data_understanding`
(duplicates, invalid values, missing values, inconsistent categories), engineer new attributes,
and save the cleaned dataset to `data/processed/`. Every step is logged with the number of rows before and after.

ــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــ

### 1. Setup
Load the raw data and create a cleaning log that records how many rows each step removes.

In [6]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/hotel_bookings.csv")
print("Raw data:", df.shape)

cleaning_log = []

def log_step(step, rows_before, rows_after, note=""):
    cleaning_log.append({
        "step": step,
        "rows_before": rows_before,
        "rows_after": rows_after,
        "removed": rows_before - rows_after,
        "note": note,
    })
    print(f"{step}: {rows_before:,} → {rows_after:,} rows (removed {rows_before - rows_after:,})")

Raw data: (119390, 32)


### 2. Duplicate records
`01_data_understanding` found 31,994 exact duplicate rows (26.8%). Before removing them we check
how they affect the data: the cancellation rate and the deposit types.

In [7]:
df_nodup = df.drop_duplicates()

print(f"Cancellation rate with duplicates:    {df['is_canceled'].mean():.1%}")
print(f"Cancellation rate without duplicates: {df_nodup['is_canceled'].mean():.1%}")

deposit_check = pd.DataFrame({
    "before": df["deposit_type"].value_counts(),
    "after": df_nodup["deposit_type"].value_counts(),
})
deposit_check["removed_%"] = ((1 - deposit_check["after"] / deposit_check["before"]) * 100).round(1)
deposit_check

Cancellation rate with duplicates:    37.0%
Cancellation rate without duplicates: 27.5%


,before,after,removed_%
deposit_type,,,
No Deposit,104641,86251,17.6
Non Refund,14587,1038,92.9
Refundable,162,107,34.0


In [8]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
log_step("Remove exact duplicates", before, len(df), 
         "No booking ID to tell copies apart; copies across train/test sets would inflate scores.")

Remove exact duplicates: 119,390 → 87,396 rows (removed 31,994)


### 3. Invalid records
A booking is invalid if it is logically impossible:
- **0 guests**: adults + children + babies = 0
- **0 nights**: weekend nights + week nights = 0
- **Negative price**: adr < 0
- **Impossible price**: adr > 1,000 (a single booking of 5,400, far above all others)

We first count each rule, then remove all records that break at least one of them.

In [9]:
guests = df["adults"] + df["children"].fillna(0) + df["babies"]
nights = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]

rules = {
    "zero_guests": guests == 0,
    "zero_nights": nights == 0,
    "negative_adr": df["adr"] < 0,
    "adr_above_1000": df["adr"] > 1000,
}

for name, mask in rules.items():
    print(f"{name:<16} {mask.sum():>5,}")

invalid = rules["zero_guests"] | rules["zero_nights"] | rules["negative_adr"] | rules["adr_above_1000"]
print(f"\nInvalid records (at least one rule): {invalid.sum():,}")

print("\nHighest daily rates:")
print(df["adr"].sort_values(ascending=False).head(3).to_string())

zero_guests        166
zero_nights        651
negative_adr         1
adr_above_1000       1

Invalid records (at least one rule): 759

Highest daily rates:
38749    5400.0
80728     510.0
11661     508.0


In [10]:
before = len(df)
df = df[~invalid].reset_index(drop=True)
log_step("Remove invalid records", before, len(df),
         "0 guests, 0 nights, negative adr, or adr of 5,400 (data-entry error).")

Remove invalid records: 87,396 → 86,637 rows (removed 759)


### 4. Missing values
Missing values after removing duplicates and invalid records. Each attribute gets its own treatment,
chosen from what the missing value means rather than a single rule for all.

In [11]:
missing_now = df.isnull().sum()
missing_now = missing_now[missing_now > 0]
pd.DataFrame({"missing": missing_now, "percent": (missing_now / len(df) * 100).round(2)})

,missing,percent
children,4,0.00
country,442,0.51
agent,11958,13.80
company,81467,94.03


| Attribute | Treatment | Reason |
| --- | --- | --- |
| `children` | fill with 0, convert to integer | only 4 rows; almost all bookings have 0 children |
| `country` | fill with "Unknown" | we do not invent a nationality |
| `agent` | replace with flag `has_agent` (1/0), drop the ID | empty = booked without a travel agent |
| `company` | replace with flag `has_company` (1/0), drop the ID | empty = not paid by a company (94% empty) |

In [12]:
df["children"] = df["children"].fillna(0).astype(int)
df["country"] = df["country"].fillna("Unknown")

df["has_agent"] = df["agent"].notna().astype(int)
df["has_company"] = df["company"].notna().astype(int)
df = df.drop(columns=["agent", "company"])

print("Missing values left:", df.isnull().sum().sum())
print(f"Bookings via an agent:   {df['has_agent'].mean():.1%}")
print(f"Bookings with a company: {df['has_company'].mean():.1%}")

log_step("Handle missing values", len(df), len(df),
         "children→0, country→'Unknown', agent/company → has_agent/has_company flags.")

Missing values left: 0
Bookings via an agent:   86.2%
Bookings with a company: 6.0%
Handle missing values: 86,637 → 86,637 rows (removed 0)


### 5. Inconsistent categories
Some categorical attributes contain an "Undefined" value. We check what it means in each attribute.

In [13]:
for col in ["meal", "market_segment", "distribution_channel"]:
    print(f"--- {col} ---")
    print(df[col].value_counts().to_string())
    print()

--- meal ---
meal
BB           67417
SC            9346
HB            9030
Undefined      485
FB             359

--- market_segment ---
market_segment
Online TA        51285
Offline TA/TO    13748
Direct           11652
Groups            4891
Corporate         4155
Complementary      682
Aviation           222
Undefined            2

--- distribution_channel ---
distribution_channel
TA/TO        68650
Direct       12803
Corporate     5001
GDS            178
Undefined        5



In [14]:
df["meal"] = df["meal"].replace("Undefined", "SC")
print(df["meal"].value_counts().to_string())

log_step("Fix inconsistent categories", len(df), len(df),
         "meal: 'Undefined' merged into 'SC' (both mean no meal package, per the dataset documentation).")

meal
BB    67417
SC     9831
HB     9030
FB      359
Fix inconsistent categories: 86,637 → 86,637 rows (removed 0)


### 6. Feature engineering
New attributes derived from existing ones:

| New attribute | Formula | Why |
| --- | --- | --- |
| `arrival_month_num` | month name → 1–12 | months as numbers can be sorted and used in models |
| `arrival_date` | year + month + day | a real date for time analysis and the data warehouse |
| `total_nights` | weekend nights + week nights | length of stay in one attribute |
| `total_guests` | adults + children + babies | size of the booking |
| `season` | month → Winter / Spring / Summer / Autumn | groups months into seasons |
| `is_family` | 1 if children or babies > 0 | families may behave differently |
| `room_changed` | 1 if assigned room ≠ reserved room | analysis only (known at check-in → leakage) |
| `booked_revenue` | adr × total_nights | value of the booking |
| `realized_revenue` | booked_revenue if not canceled, else 0 | revenue actually earned (uses the outcome → not a model input) |

In [15]:
months = ["January", "February", "March", "April", "May", "June",
          "July", "August", "September", "October", "November", "December"]
df["arrival_month_num"] = df["arrival_date_month"].map({m: i + 1 for i, m in enumerate(months)})

df["arrival_date"] = pd.to_datetime(dict(year=df["arrival_date_year"],
                                         month=df["arrival_month_num"],
                                         day=df["arrival_date_day_of_month"]))

df["total_nights"] = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
df["total_guests"] = df["adults"] + df["children"] + df["babies"]

def to_season(month):
    if month in (12, 1, 2):
        return "Winter"
    if month in (3, 4, 5):
        return "Spring"
    if month in (6, 7, 8):
        return "Summer"
    return "Autumn"

df["season"] = df["arrival_month_num"].map(to_season)

df["is_family"] = ((df["children"] + df["babies"]) > 0).astype(int)
df["room_changed"] = (df["reserved_room_type"] != df["assigned_room_type"]).astype(int)

df["booked_revenue"] = df["adr"] * df["total_nights"]
df["realized_revenue"] = np.where(df["is_canceled"] == 0, df["booked_revenue"], 0.0)

new_cols = ["arrival_month_num", "arrival_date", "total_nights", "total_guests", "season",
            "is_family", "room_changed", "booked_revenue", "realized_revenue"]
df[new_cols].head()

,arrival_month_num,arrival_date,total_nights,total_guests,season,is_family,room_changed,booked_revenue,realized_revenue
0,7,2015-07-01,1,1,Summer,0,1,75.0,75.0
1,7,2015-07-01,1,1,Summer,0,0,75.0,75.0
2,7,2015-07-01,2,2,Summer,0,0,196.0,196.0
3,7,2015-07-01,2,2,Summer,0,0,214.0,214.0
4,7,2015-07-01,2,2,Summer,0,0,206.0,206.0


In [16]:
print(f"Average nights per booking: {df['total_nights'].mean():.2f}")
print(f"Average guests per booking: {df['total_guests'].mean():.2f}")
print(f"Family bookings:            {df['is_family'].mean():.1%}")
print(f"Room changed at check-in:   {df['room_changed'].mean():.1%}")
print()
print(df["season"].value_counts().to_string())
print()
booked, realized = df["booked_revenue"].sum(), df["realized_revenue"].sum()
print(f"Booked revenue:   €{booked:,.0f}")
print(f"Realized revenue: €{realized:,.0f}")
print(f"Lost to cancellations: €{booked - realized:,.0f} ({(booked - realized) / booked:.1%})")

log_step("Feature engineering", len(df), len(df), "Added " + ", ".join(new_cols) + ".")

Average nights per booking: 3.65
Average guests per booking: 2.03
Family bookings:            10.5%
Room changed at check-in:   14.6%

season
Summer    28903
Spring    23604
Autumn    18417
Winter    15713

Booked revenue:   €34,446,255
Realized revenue: €22,967,612
Lost to cancellations: €11,478,643 (33.3%)
Feature engineering: 86,637 → 86,637 rows (removed 0)


### 7. Cleaning log
Every preprocessing step with the number of rows before and after.

In [17]:
log_df = pd.DataFrame(cleaning_log)
log_df

,step,rows_before,rows_after,removed,note
0,Remove exact duplicates,119390,87396,31994,No booking ID to tell copies apart; copies acr...
1,Remove invalid records,87396,86637,759,"0 guests, 0 nights, negative adr, or adr of 5,..."
2,Handle missing values,86637,86637,0,"children→0, country→'Unknown', agent/company →..."
3,Fix inconsistent categories,86637,86637,0,meal: 'Undefined' merged into 'SC' (both mean ...
4,Feature engineering,86637,86637,0,"Added arrival_month_num, arrival_date, total_n..."


### 8. Final checks
Verify the cleaned dataset before saving: size, missing values, data types and duplicates.

In [18]:
print("Shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print()
print(df.dtypes.value_counts().to_string())

assert df.isnull().sum().sum() == 0, "There are still missing values"
assert len(df) == 86637, "Unexpected number of rows"

Shape: (86637, 41)
Missing values: 0
Duplicate rows: 17

int64             24
object            13
float64            3
datetime64[ns]     1


The 17 "duplicate" rows are **not** real duplicates: they became identical only after replacing
the agent and company IDs with the `has_agent` / `has_company` flags. In the original data they are
different bookings (different agents or companies), so they are kept.

### 9. Save the cleaned dataset

In [19]:
df.to_csv("../data/processed/hotel_bookings_clean.csv", index=False)
log_df.to_csv("../data/processed/cleaning_log.csv", index=False)
print("Saved:", df.shape)

Saved: (86637, 41)


ــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــ


## Summary

| | Rows | Columns |
| --- | --- | --- |
| Raw dataset | 119,390 | 32 |
| Cleaned dataset | **86,637** | **41** |

- **31,994 exact duplicates removed.** 93% of "Non Refund" bookings were duplicates; removing them
  changed the cancellation rate from 37.0% to 27.5%.
- **759 invalid records removed:** 0 guests, 0 nights, a negative rate, and a rate of €5,400.
- **No missing values left:** children → 0, country → "Unknown", agent and company → `has_agent` / `has_company` flags.
- **Categories fixed:** meal "Undefined" merged into "SC".
- **9 new attributes**, including `booked_revenue`: cancellations cost the two hotels **€11.5M (33.3%)** of booked revenue.

**Not done here on purpose:** encoding, scaling, grouping rare countries and excluding leakage attributes
(`reservation_status`, `reservation_status_date`, `room_changed`, `realized_revenue`) are done inside each model
notebook, because each algorithm needs a different form and scaling must be learned from the training set only.

**Next step:** `03_data_warehouse.ipynb` — load the cleaned data into a star schema.